<a href="https://colab.research.google.com/github/abhsrivastava/hugging_face_transformers/blob/main/Gradio_Blocks_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Today we will build a UI App which uses our fine tuned model via Gradio Blocks Framework

In [1]:
# Install dependencies
%pip install -q transformers gradio

print(f'✅ Installed Dependencies Successfully!')

✅ Installed Dependencies Successfully!


In [2]:
# Import packages
from transformers import pipeline
import gradio as gr
import warnings
import pandas as pd
warnings.filterwarnings('ignore')

print(f'✅ Imports successful!')


✅ Imports successful!


In [3]:
classifier = pipeline("text-classification", model="abhishes/novapay-sentiment")

print(f'✅ Classifier loaded successfully')

config.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

✅ Classifier loaded successfully


In [4]:
# verify versions

print(f'Gradio Version: {gr.__version__}')

Gradio Version: 6.20.0


In [5]:
# create CSS

css = """
.history-table {
    width: 100%;
    border-collapse: collapse;
    font-size: 16px;
}

.history-table th {
    background: #171717;
    color: white;
    text-align: left;
    padding: 12px;
    border: 1px solid #444;
}

.history-table td {
    padding: 12px;
    border: 1px solid #444;
}

.history-table tr:nth-child(even) {
    background: rgba(255, 255, 255, 0.04);
}

.empty-history {
    color: #999;
    padding: 16px;
}
"""

In [6]:
from html import escape

def render_history(records):
    if not records:
        return "<p style='color:#999;'>No classifications yet.</p>"

    rows = ""

    # Newest record first
    for record in reversed(records):
        message = escape(str(record["Message"]))
        label = escape(str(record["Label"]))
        score = escape(str(record["Score"]))

        rows += f"""
        <tr>
            <td>{message}</td>
            <td>{label}</td>
            <td>{score}</td>
        </tr>
        """

    return f"""
    <div style="overflow-x:auto;">
        <table style="
            width:100%;
            border-collapse:collapse;
            font-size:16px;
        ">
            <thead>
                <tr style="background:#181818;">
                    <th style="padding:12px; border:1px solid #444; text-align:left;">
                        Message
                    </th>
                    <th style="padding:12px; border:1px solid #444; text-align:left;">
                        Label
                    </th>
                    <th style="padding:12px; border:1px solid #444; text-align:left;">
                        Score
                    </th>
                </tr>
            </thead>

            <tbody>
                {rows}
            </tbody>
        </table>
    </div>

    <style>
        table td {{
            padding: 12px;
            border: 1px solid #444;
            text-align: left;
        }}

        table tbody tr:nth-child(even) {{
            background: rgba(255, 255, 255, 0.05);
        }}
    </style>
    """

In [7]:
# write classification function

def classify(text, threshold, history):
  history = history or []
  if not text or not text.strip():
    return (
        {"no input": 1.0},
        gr.update(visible=False, value=""),
        render_history(history),
        history
      )

  result = classifier(text)[0]
  label = ("Positive" if result["label"] == "LABEL_1" else "Negative")

  score = float(result["score"])
  label_dict = {label: score}

  new_record = {
    "Message": text[:60] + ("..." if len(text) > 60 else ""),
    "Label": label,
    "Score": f"{result['score']:0.4f}"
  }

  updated_history = history + [new_record]
  displayed_history = render_history(updated_history)

  if score < threshold:
    warning = (f"⚠️ Low confidence " f"({score:.3f} < {threshold:.2f}) — " "Manual review recommended")
    warning_update = gr.update(visible=True, value=warning)
  else:
    warning_update = gr.update(visible=False, value="")

  return (label_dict, warning_update, displayed_history, updated_history)



In [8]:
with gr.Blocks("NovaPay Sentiment Analyzer v 1.0", css=css) as demo:
  gr.Markdown("# NovaPay Sentiment Analyzer v1.0")

  #history
  history_state = gr.State(value=lambda: [])

  with gr.Row():
    with gr.Column():
      input_text = gr.Textbox(label="Customer Message: ", placeholder="Type a NovaPay customer message...", lines=5)
      threshold_slider = gr.Slider(minimum=0.5, maximum=0.99, value=0.8, step=0.01, label="Confidence Threshold")
      submit_button = gr.Button("🔎 Analyze Message", variant="primary")
    with gr.Column():
      output_label = gr.Label(label="Sentiment Result")
      warning_box = gr.Textbox(label="⚠️ Alert", visible=False, interactive=False)
  with gr.Row():
    gr.Markdown("### 📊 Classfication History")
  with gr.Row():
    history_display = gr.HTML(value=render_history([]))
    submit_button.click(fn=classify, inputs=[input_text, threshold_slider, history_state], outputs=[output_label, warning_box, history_display, history_state])

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7a3b45222148ec31e0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:
demo.close()

Closing server running on port: 7861
